# 01 - Data Cleaning
This notebook handles the sanitization of our raw dataset. We will:
1. Load the raw data.
2. Drop corrupted or completely empty rows.
3. Check EVERY SINGLE COLUMN in the dataset.
4. Fix data types (convert Dates and Times to standard datetime objects).
5. Enforce Numeric types for counts (Enrolled, Present, Semester, Lecture).
6. Clean all categorical text fields (remove trailing spaces).
7. Calculate the true `Attendance_Percentage`.
8. Export the clean data to the `processed/` folder.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load the raw 500-line dataset
raw_data_path = '../../../data/raw/raw_attendance.csv'
df = pd.read_csv(raw_data_path)

print(f"Initial shape: {df.shape}")
print("\n--- Columns Present ---")
print(df.columns.tolist())
df.head()

Initial shape: (500, 22)

--- Columns Present ---
['Date', 'Day_of_Week', 'Lecture_Number', 'Start_Time', 'End_Time', 'Subject', 'Faculty_ID', 'Semester', 'Branch', 'Section', 'Classroom', 'Total_Enrolled', 'Students_Present', 'Attendance_Percentage', 'Previous_Lecture_Attendance', 'Gap_Since_Previous_Lecture', 'Practical_Theory', 'Internal_Test_Week', 'Assignment_Due', 'Holiday_Before_After', 'Weather', 'Special_Event']


,Date,Day_of_Week,Lecture_Number,Start_Time,End_Time,Subject,Faculty_ID,Semester,Branch,Section,...,Students_Present,Attendance_Percentage,Previous_Lecture_Attendance,Gap_Since_Previous_Lecture,Practical_Theory,Internal_Test_Week,Assignment_Due,Holiday_Before_After,Weather,Special_Event
0,04-04-2026,Saturday,1,8.30 AM,9.15 AM,MAD Practical,SSP_ASP,3,MCA,A&B,...,27,13.24,NaN,NaN,Practical,No,No,No,Sunny,No
1,04-04-2026,Saturday,2,9.15 AM,10.15 AM,MAD Practical,SSP_ASP,3,MCA,A&B,...,20,9.80,NaN,NaN,Practical,No,No,No,Sunny,No
2,04-04-2026,Saturday,4,11.15 AM,12.15 PM,Mini Project,SP,3,MCA,A&B,...,41,20.10,NaN,NaN,Practical,No,No,No,Sunny,No
3,06-04-2026,Monday,1,8.30 AM,9.15 AM,Mobile Application Development,SSP,3,MCA,A&B,...,36,17.65,NaN,NaN,Theory,No,No,No,Sunny,No
4,06-04-2026,Monday,2,9.15 AM,10.15 AM,Mobile Application Development,SSP,3,MCA,A&B,...,43,21.08,NaN,NaN,Theory,No,No,No,Sunny,No


### Step 1: Handling Missing or Corrupted Data
We drop any rows that don't have a valid Date, as these are usually Excel artifacts.

In [2]:
df.dropna(subset=['Date'], inplace=True)
print(f"Shape after dropping null dates: {df.shape}")

# Ensure NO missing values across ANY column. 
# If there are any, we forward-fill them as a safety fallback.
missing_counts = df.isnull().sum()
if missing_counts.sum() > 0:
    print("\nMissing values detected in some columns. Forward filling...")
    df.ffill(inplace=True)
else:
    print("\nZero missing values detected across all columns!")

Shape after dropping null dates: (500, 22)

Missing values detected in some columns. Forward filling...


### Step 2: Fixing Data Types (Dates & Times)
We will format the `Date`, `Start_Time`, and `End_Time` columns.

In [3]:
# Convert Date column to datetime
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

# Convert Start_Time and End_Time to standard time objects
df['Start_Time'] = pd.to_datetime(df['Start_Time'], format='%I.%M %p').dt.time
df['End_Time'] = pd.to_datetime(df['End_Time'], format='%I.%M %p').dt.time

df[['Date', 'Start_Time', 'End_Time']].head()

,Date,Start_Time,End_Time
0,2026-04-04,08:30:00,09:15:00
1,2026-04-04,09:15:00,10:15:00
2,2026-04-04,11:15:00,12:15:00
3,2026-04-06,08:30:00,09:15:00
4,2026-04-06,09:15:00,10:15:00


### Step 3: Enforcing Numeric Data Types
We must ensure that numerical columns are strictly cast as integers to prevent calculation errors.

In [4]:
numeric_columns = ['Lecture_Number', 'Semester', 'Total_Enrolled', 'Students_Present']

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], downcast='integer', errors='coerce').fillna(0).astype(int)

print("Numeric columns strictly cast to integers.")

Numeric columns strictly cast to integers.


### Step 4: Cleaning Text Categories
We iterate dynamically through every remaining column. If it is an 'object' (text), we strip any accidental trailing spaces.

In [5]:
# Get all columns that are still text (objects)
categorical_columns = df.select_dtypes(include=['object']).columns

for col in categorical_columns:
    # Skip the time columns we just created
    if col not in ['Start_Time', 'End_Time']:
        df[col] = df[col].astype(str).str.strip()

print("All textual categorical columns comprehensively cleaned.")

All textual categorical columns comprehensively cleaned.


### Step 5: Calculating Attendance Percentage & Logical Checks
We calculate `Attendance_Percentage`.

In [6]:
# Calculate the Percentage Target Variable
df['Attendance_Percentage'] = (df['Students_Present'] / df['Total_Enrolled']) * 100
df['Attendance_Percentage'] = df['Attendance_Percentage'].round(2)

# Logical Gate: Assert that Present <= Enrolled
assert (df['Students_Present'] <= df['Total_Enrolled']).all(), "CRITICAL ERROR: Students present exceeds total enrolled capacity!"
print("Logical Gate Passed: Students_Present is always <= Total_Enrolled.")

Logical Gate Passed: Students_Present is always <= Total_Enrolled.


### Step 6: Sorting & Exporting
Sorted chronologically before EDA phase.

In [7]:
# Sort purely by Date and Start Time
df.sort_values(by=['Date', 'Start_Time'], inplace=True)

import os
out_dir = '../../../data/processed'
os.makedirs(out_dir, exist_ok=True)

out_path = f"{out_dir}/attendance_cleaned.csv"
df.to_csv(out_path, index=False)
print(f"Cleaned dataset successfully saved to: {out_path}")

Cleaned dataset successfully saved to: ../../../data/processed/attendance_cleaned.csv
